In [10]:
!pip install -q langchain-core requests langchain_huggingface

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
import requests
import os


In [12]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [13]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [14]:
multiply.name
multiply.description
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [24]:
from google.colab import userdata

llm = HuggingFaceEndpoint(
    repo_id="google/gemma-4-31B-it",
    huggingfacehub_api_token=userdata.get("HUGGINGFACEHUB_API_TOKEN")
)

model = ChatHuggingFace(llm=llm)

In [42]:
model.invoke([HumanMessage(content='hi')])

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9be5-2bbf-7201-9382-d626ebadfbd2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24})

In [29]:
llm_with_tools = model.bind_tools([multiply])

In [30]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you for asking! How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 81, 'total_tokens': 100}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9be2-35b1-7451-87d9-b161fcd1080a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 19, 'total_tokens': 100})

In [44]:
query = HumanMessage('can you multiply 3 with 1000')
messages = [query]

In [45]:
result = llm_with_tools.invoke(messages)

In [46]:
messages.append(result)

In [47]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-911e55b53ca60c84', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 88, 'total_tokens': 106}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9be5-a08c-7763-b55d-ae7eaad87c01-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-911e55b53ca60c84', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 88, 'output_tokens': 18, 'total_tokens': 106})]

In [48]:
tool_result = multiply.invoke(result.tool_calls[0])

In [49]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-911e55b53ca60c84')

In [50]:
messages.append(tool_result)

In [51]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-911e55b53ca60c84', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 88, 'total_tokens': 106}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9be5-a08c-7763-b55d-ae7eaad87c01-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-911e55b53ca60c84', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 88, 'output_tokens': 18, 'total_tokens': 106}),
 ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-911e55b53ca60c84')]

In [41]:
llm_with_tools.invoke(messages).content

'3 multiplied by 1000 is 3000.'

In [52]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [53]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [54]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780704002,
 'time_last_update_utc': 'Sat, 06 Jun 2026 00:00:02 +0000',
 'time_next_update_unix': 1780790402,
 'time_next_update_utc': 'Sun, 07 Jun 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2231}

In [55]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

# tool binding

In [57]:
llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [58]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [59]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [60]:
ai_message = llm_with_tools.invoke(messages)

In [61]:
messages.append(ai_message)

In [62]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'chatcmpl-tool-a254d37180923f32',
  'type': 'tool_call'}]

In [63]:
import json
for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [64]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'chatcmpl-tool-a254d37180923f32', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 178, 'total_tokens': 205}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9be8-8661-7cb1-9216-f4b50c0d0ee7-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'chatcmpl-tool-a254d37180923f32', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens': 27, 'total_tokens': 205}),
 ToolMessage(content='{"result

In [65]:
llm_with_tools.invoke(messages).content

''

In [78]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# Define a prompt template for the agent
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that can use tools."),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# Create the agent using the chat model (model) and tools
agent = create_agent(
    model=model,
    tools=[get_conversion_factor, convert],
    system_prompt="You are a helpful assistant that can use tools."
)

# Initialize the Agent Executor
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Convert 5 kilometers to meters"
            }
        ]
    }
)

In [81]:
user_query = "Hi how are you?"

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    }
)
response

{'messages': [HumanMessage(content='Hi how are you?', additional_kwargs={}, response_metadata={}, id='fb1d9bbe-c686-43a6-bd44-3279aabc463e'),
  AIMessage(content="I'm doing well, thank you for asking! How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 181, 'total_tokens': 200}, 'model_name': 'google/gemma-4-31B-it', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9c0f-881a-7c10-88ce-132fd2b79066-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 181, 'output_tokens': 19, 'total_tokens': 200})]}